Scraping

In [ ]:
import warnings
import pandas as pd
import re
import time
import json

from ddgs import DDGS
import trafilatura

from newspaper import Article

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By

warnings.filterwarnings("ignore")


class FinancialNewsScraper:

    def __init__(self):

        self.headers = {
            "User-Agent":
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"
        }

        self.domains = [

            "cnbcindonesia.com",
            "kompas.com",
            "kumparan.com",
            "detik.com",
            "jpnn.com",
            "tempo.co",
            "viva.co.id",
            "kontan.co.id",
            "bisnis.com",
            "stockwatch.id",
            "investor.id",
            "idxchannel.com"
        ]

        self.bad_patterns = [

            "/tag/",
            "/tags/",
            "/topic/",
            "/topics/",
            "/search/",
            "/author/",
            "/kategori/",
            "/label/",
            "/page/",
            "/arsip/"
        ]

    # =====================================================
    # CLEANING
    # =====================================================

    def clean_text(self, text):

        if text is None:
            return ""

        text = text.lower()
        text = re.sub(r"http\S+", " ", text)
        text = re.sub(r"www\S+", " ", text)
        text = re.sub(r"\S+@\S+", " ", text)
        text = re.sub(r"\s+", " ", text)
        return text.strip()

    # =====================================================
    # FILTER URL
    # =====================================================

    def is_valid_url(self, url):

        url_lower = url.lower()

        if not any(
            domain in url_lower
            for domain in self.domains
        ):
            return False

        if any(
            bad in url_lower
            for bad in self.bad_patterns
        ):
            return False
        return True

    # =====================================================
    # SEARCH NEWS LINK
    # =====================================================

    def get_news_links(
        self,
        keyword,
        category,
        max_results=50
    ):

        links = []

        if category == "market":

            queries = [
                f'"{keyword}" saham',
                f'"{keyword}" emiten',
                f'"{keyword}" bisnis',
                f'"{keyword}" site:cnbcindonesia.com',
                f'"{keyword}" site:kontan.co.id',
                f'"{keyword}" site:bisnis.com'
            ]

        elif category == "macro":

            queries = [
                f'"{keyword}" ekonomi',
                f'"{keyword}" indonesia',
                f'"{keyword}" pasar modal',
                f'"{keyword}" site:cnbcindonesia.com',
                f'"{keyword}" site:kontan.co.id',
                f'"{keyword}" site:bisnis.com'
            ]

        elif category == "geopolitics":

            queries = [
                f'"{keyword}" ekonomi',
                f'"{keyword}" pasar global',
                f'"{keyword}" investasi',
                f'"{keyword}" site:cnbcindonesia.com',
                f'"{keyword}" site:kontan.co.id',
                f'"{keyword}" site:bisnis.com'
            ]

        else:

            queries = [keyword]

        with DDGS() as ddgs:

            for query in queries:

                print(f"\nQuery: {query}")

                try:

                    results = ddgs.text(
                        query,
                        region="id-id",
                        safesearch="off",
                        max_results=max_results
                    )

                    for r in results:

                        try:
                            url = r["href"]
                            if self.is_valid_url(url):
                                links.append(url)
                        except:
                            pass

                except Exception as e:
                    print(f"DDGS ERROR: {e}")
        links = list(set(links))

        return links

    # =====================================================
    # EXTRACT VIA TRAFILATURA
    # =====================================================

    def extract_with_trafilatura(self, url):

        try:

            downloaded = trafilatura.fetch_url(url)

            if downloaded is None:
                return None

            metadata_json = trafilatura.extract(
                downloaded,
                output_format="json",
                with_metadata=True
            )

            if metadata_json is None:
                return None

            article = json.loads(metadata_json)

            return {
                "title":
                article.get("title", ""),
                "text":
                article.get("text", ""),
                "date":
                article.get("date", "")
            }

        except Exception as e:

            print(f"TRAFILATURA ERROR: {e}")

            return None

    # =====================================================
    # EXTRACT VIA NEWSPAPER3K
    # =====================================================

    def extract_with_newspaper(self, url):

        try:

            article = Article(url)

            article.download()

            article.parse()

            return {
                "title":
                article.title,
                "text":
                article.text,
                "date":
                str(article.publish_date)
                if article.publish_date else ""
            }

        except Exception as e:
            print(f"NEWSPAPER ERROR: {e}")
            return None

    # =====================================================
    # EXTRACT VIA SELENIUM
    # =====================================================

    def extract_with_selenium(self, url):
        driver = None

        try:

            options = webdriver.ChromeOptions()

            options.add_argument("--headless")

            options.add_argument("--disable-blink-features=AutomationControlled")

            options.add_argument(
                "user-agent=Mozilla/5.0"
            )

            driver = webdriver.Chrome(
                service=Service(
                    ChromeDriverManager().install()
                ),
                options=options
            )

            driver.get(url)

            time.sleep(5)

            body = driver.find_element(
                By.TAG_NAME,
                "body"
            ).text

            title = driver.title

            return {
                "title": title,
                "text": body,
                "date": ""
            }

        except Exception as e:

            print(f"SELENIUM ERROR: {e}")

            return None

        finally:
            if driver:
                driver.quit()

    # =====================================================
    # SCRAPE ARTICLES
    # =====================================================

    def scrape_articles(
        self,
        urls,
        keyword,
        category
    ):

        data = []

        for idx, url in enumerate(urls):
            try:
                print(
                    f"\n[{idx+1}/{len(urls)}]"
                )

                print(url)

                title = ""
                text = ""
                date = ""

                # =========================================
                # 1. TRAFILATURA
                # =========================================

                result = self.extract_with_trafilatura(
                    url
                )

                if result:
                    title = result["title"]
                    text = result["text"]
                    date = result["date"]

                    print(
                        f"TRAFILATURA: {len(text)} karakter"
                    )

                # =========================================
                # 2. NEWSPAPER FALLBACK
                # =========================================

                if len(text) < 500:

                    print(
                        "Fallback -> newspaper3k"
                    )

                    result = self.extract_with_newspaper(
                        url
                    )

                    if result:
                        title = result["title"]
                        text = result["text"]
                        date = result["date"]

                        print(
                            f"NEWSPAPER: {len(text)} karakter"
                        )

                # =========================================
                # 3. SELENIUM FALLBACK
                # =========================================

                if len(text) < 500:

                    print(
                        "Fallback -> selenium"
                    )

                    result = self.extract_with_selenium(
                        url
                    )

                    if result:
                        title = result["title"]
                        text = result["text"]
                        date = result["date"]
                        print(
                            f"SELENIUM: {len(text)} karakter"
                        )

                # =========================================
                # CLEAN TEXT
                # =========================================

                text = self.clean_text(text)

                # =========================================
                # FILTER PANJANG
                # =========================================

                if len(text) < 500:
                    print(
                        "SKIP: kurang dari 500 karakter"
                    )
                    continue

                # =========================================
                # FILTER RELEVANSI
                # =========================================

                keyword_count = self.count_keyword_occurrence(
                    text,
                    keyword
                )

                print(
                    f"Keyword count: {keyword_count}"
                )

                if category == "market":
                    if keyword_count < 2:
                        continue

                elif category == "geopolitics":
                    match_count = self.geopolitic_match(
                        text,
                        keyword
                    )
                    if match_count < 2:
                        continue
                else:
                    if keyword_count < 1:
                        continue

                # =========================================
                # SAVE
                # =========================================

                data.append({
                    "domain":
                    category,

                    "keyword":
                    keyword,

                    "title":
                    title,

                    "date":
                    date,

                    "url":
                    url,

                    "text":
                    text,

                    "source":
                    url.split("/")[2]
                    .replace("www.", "")
                })
                print(
                    f"OK -> {len(text)} karakter"
                )
                time.sleep(1)
            except Exception as e:
                print(f"ERROR: {e}")
        return data

    def count_keyword_occurrence(
            self,
            text,
            keyword
        ):
            text = text.lower()
            words = keyword.lower().split()
            total = 0
            for word in words:
                total += text.count(word)
            return total

    def geopolitic_match(
            self,
            text,
            keyword
        ):
            text = text.lower()
            words = keyword.lower().split()
            found = 0
            for word in words:
                if word in text:
                    found += 1
            return found

# =====================================================
# KEYWORDS
# =====================================================

keywords = {
    "market": [
        # Emiten
        "BBCA",
        "BBRI",
        "BMRI",
        "BBNI",
        "TLKM",
        "ASII",
        "ANTM",
        "PTBA",
        "ADRO",
        "INDF",
        "ICBP",
        "KLBF",
        "PGAS",
        "UNTR",
        "MEDC",
        "GOTO",
        "AMMN",
        "BRPT",
        "CPIN",
        "SMGR",

        # Crypto
        "bitcoin", "ethereum", "solana", "ripple", "dogecoin",

        # Komoditas
        "emas", "perak", "minyak dunia", "minyak mentah", "batubara", "nikel", "tembaga", "timah", "gas alam", "cpo",

        # Pasar Modal
        "IHSG",
        "LQ45",
        "IDX",
        "BEI",
        "dividen",
        "buyback saham",
        "right issue",
        "IPO Indonesia"
    ],

    "macro": [
        "inflasi indonesia",
        "suku bunga BI",
        "BI Rate",
        "Bank Indonesia",
        "rupiah",
        "nilai tukar rupiah",
        "pertumbuhan ekonomi indonesia",
        "GDP Indonesia",
        "neraca perdagangan",
        "ekspor indonesia",
        "impor indonesia",
        "cadangan devisa",
        "pengangguran indonesia",
        "PMI manufaktur Indonesia",
        "Fed Rate",
        "ECB Rate",
        "ekonomi global"
        "deflasi indonesia",
        "daya beli masyarakat",
        "konsumsi rumah tangga",
        "APBN",
        "utang pemerintah",
        "yield obligasi",
        "SBN",
        "surat berharga negara",
    ],

    "geopolitics": [
        "Iran Israel",
        "Timur Tengah",
        "Selat Hormuz",
        "Laut Merah",
        "Houthi",
        "Rusia Ukraina",
        "Rusia NATO",
        "sanksi Rusia",
        "invasi Ukraina",
        "AS China",
        "Amerika Serikat China",
        "perang dagang",
        "perang tarif",
        "tarif impor Amerika",
        "Taiwan China",
        "Taiwan Tiongkok",
        "OPEC",
        "krisis energi",
        "Laut China Selatan",
        "BRICS",
        "G7"
    ]
}

# =====================================================
# MAIN
# =====================================================

scraper = FinancialNewsScraper()

all_articles = []

for category, words in keywords.items():
    print(
        f"\n{'='*70}"
    )
    print(
        f"CATEGORY: {category}"
    )
    print(
        f"{'='*70}"
    )
    for keyword in words:
        print(
            f"\nKeyword: {keyword}"
        )
        links = scraper.get_news_links(
            keyword=keyword,
            category=category,
            max_results=30
        )
        print(
            f"Total URL: {len(links)}"
        )

        articles = scraper.scrape_articles(
            urls=links,
            keyword=keyword,
            category=category
        )
        all_articles.extend(
            articles
        )

# =====================================================
# SAVE DATASET
# =====================================================

df = pd.DataFrame(all_articles)

if len(df) > 0:

    # Hapus duplicate URL
    df.drop_duplicates(
        subset=["url"],
        inplace=True
    )

    # Hapus duplicate text
    df.drop_duplicates(
        subset=["text"],
        inplace=True
    )

    # Sort tanggal
    if "date" in df.columns:
        try:
            df = df.sort_values(
                by="date",
                ascending=False
            )
        except:
            pass
    print(
        f"\nTotal Artikel Bersih: {len(df)}"
    )

    df.to_csv(
        "dataset_finansial_indonesia_clean_new.csv",
        index=False,
        encoding="utf-8-sig"
    )
    print(
        "\nDataset berhasil disimpan"
    )
    print(
        "dataset_finansial_indonesia_clean_new.csv"
    )
else:
    print(
        "\nTidak ada artikel berhasil discrape"
    )

Labelling

In [ ]:
import os
import json
import pandas as pd
import time
from openai import OpenAI
from tqdm import tqdm

# ==========================================
# REVISI: API KEY & CLIENT OPENROUTER
# ==========================================
OPENROUTER_API_KEY = "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"

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY
)

# ==========================================
# REVISI: MODEL NAME (GPT-4o-mini)
# ==========================================
MODEL_NAME = "openai/gpt-4o-mini"

# ==========================================
# LABEL MAP
# ==========================================
label_map = {
    "bearish": 0,
    "neutral": 1,
    "bullish": 2
}

SYSTEM_PROMPT = """
Anda adalah analis pasar modal Indonesia.

Tugas:
Berikan sentimen terhadap DAMPAK berita terhadap pasar keuangan dan investasi.

Kategori berita:
1. market
   - saham
   - emiten
   - indeks
   - komoditas
   - crypto

2. macro
   - inflasi
   - suku bunga
   - kurs
   - GDP
   - ekonomi Indonesia

3. geopolitics
   - perang
   - konflik negara
   - tarif impor
   - sanksi ekonomi
   - ketegangan global

Aturan Labeling (Pilih salah satu dari 3 label wajib ini):
- Bullish : Jika berita meningkatkan sentimen pasar, mendukung kenaikan harga aset, atau mendukung pertumbuhan ekonomi.
- Bearish : Jika berita meningkatkan risiko pasar, menekan harga aset, atau menekan ekonomi/investasi di Indonesia.
- Neutral : Jika dampak berita tidak jelas bagi pasar, bersifat sekadar informatif, atau memiliki dampak campuran (pro & kontra seimbang).

PENTING:

Jangan memilih Neutral hanya karena artikel bersifat berita.

Neutral hanya digunakan jika:

1. tidak ada dampak ekonomi
2. tidak ada dampak pasar
3. dampaknya benar-benar seimbang

Jika terdapat indikasi positif terhadap:
- saham
- investasi
- ekonomi
- perdagangan
- rupiah
- komoditas

maka pilih Bullish.

Jika terdapat indikasi negatif terhadap:
- saham
- investasi
- ekonomi
- perdagangan
- rupiah
- komoditas

maka pilih Bearish.

Kembalikan JSON valid dengan format persis seperti ini:
{
  "sentiment": "bullish",
  "reason": "..."
}
"""

def label_article(category, keyword, title, article_text):
    # REVISI: Batas pemotongan teks ditingkatkan ke 4000 karakter agar konteks berita makro/pasar utuh
    truncated_text = article_text[:2500]

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },
                {
                    "role": "user",
                    "content": f"""
                    Kategori: {category}
                    Topik: {keyword}
                    Judul: {title}
                    Artikel: {truncated_text}
                    """
                }
            ]
        )

        result = response.choices[0].message.content

        # REVISI: Membersihkan tagging markdown penulisan json yang terkadang ikut terbawa
        result = result.replace("```json", "").replace("```", "").strip()

        data = json.loads(result)

        sentiment = data.get("sentiment", "neutral").lower().strip()
        reason = data.get("reason", "")

        if sentiment not in ["bullish", "bearish", "neutral"]:
            sentiment = "neutral"

        return sentiment, reason

    except Exception as e:
        print(f"\nERROR pada artikel '{title}': {e}")
        return "neutral", "parse_error"

# ==========================================
# LOAD DATASET
# ==========================================
# Pastikan nama file input Anda sudah sesuai di direktori kerja
file_input = "dataset_finansial_indonesia.csv"
df = pd.read_csv(file_input)

# Data Pre-processing aman dari nilai NaN
df['text'] = df['text'].fillna('')
df['title'] = df['title'].fillna('')
df['domain'] = df['domain'].fillna('market')
df['keyword'] = df['keyword'].fillna('umum')

SAVE_EVERY = 25

sentiments = []
labels = []
reasons = []
keywords_used = []

# Loop proses pelabelan
for index, row in tqdm(df.iterrows(), total=len(df), desc="Proses Pelabelan GPT-4o-mini"):

    title = str(row["title"])
    text = str(row["text"])

    sentiment, reason = label_article(
        row["domain"],
        row["keyword"],
        title,
        text
    )

    sentiments.append(sentiment)
    labels.append(label_map.get(sentiment, 1))
    reasons.append(reason)
    keywords_used.append(row["keyword"])

    # ==========================
    # CHECKPOINT PER 25 DATA
    # ==========================
    if len(sentiments) % SAVE_EVERY == 0:
        temp_df = df.iloc[:len(sentiments)].copy()
        temp_df["sentiment"] = sentiments
        temp_df["label"] = labels
        temp_df["reason"] = reasons

        checkpoint_file = f"checkpoint_{len(sentiments)}.csv"
        temp_df.to_csv(
            checkpoint_file,
            index=False,
            encoding="utf-8-sig"
        )
        print(f"\n[CHECKPOINT] Berhasil menyimpan {len(sentiments)} data ke {checkpoint_file}")

    # Delay kecil untuk menjaga stabilitas kuota rate-limit OpenRouter Anda
    time.sleep(0.5)

# ==========================================
# FINALISASI DATASET & EKSPOR
# ==========================================
df["sentiment"] = sentiments
df["label"] = labels
df["reason"] = reasons

print("\n--- DISTRIBUSI DOMAIN KATEGORI ---")
print(df["domain"].value_counts())

# Pembuatan teks gabungan terstruktur untuk kebutuhan training IndoBERT
df["input_text"] = (
    "[" + df["domain"].astype(str).str.upper() + "] " +
    df["title"].astype(str) + " " +
    df["text"].astype(str)
)

df.rename(
    columns={"domain": "category"},
    inplace=True
)

df["llm_model"] = MODEL_NAME

# Membuat file distribusi statistik sentimen berdasarkan kata kunci (keyword)
distribution = (
    df.groupby(["keyword", "sentiment"])
    .size()
    .reset_index(name="count")
)
distribution.to_csv(
    "distribution_sentiment.csv",
    index=False
)

# REVISI: Menyimpan hasil akhir ke CSV menggunakan delimiter titik koma (;) sesuai instruksi Anda
output_final = "dataset_finansial_labeled.csv"
df.to_csv(
    output_final,
    sep=";",
    index=False,
    encoding="utf-8-sig"
)

print(f"\nProses Selesai! File output final disimpan di: {output_final}")

Training

In [ ]:
import os
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    f1_score,
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────
# 0. KONFIGURASI
# ─────────────────────────────────────────────
CONFIG = {
    # Data
    "data_path": "dataset_finansial_labeled (2).csv",
    "text_col": "text",
    "label_col": "label",
    "test_size": 0.15,
    "val_size": 0.15,
    "random_seed": 42,

    # Model
    "pretrained_model": "indobenchmark/indobert-base-p1",
    "num_labels": 3,
    "max_length": 256,

    # Training
    "epochs": 5,
    "batch_size": 16,
    "learning_rate": 2e-5,
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "gradient_clip": 1.0,
    "early_stopping_patience": 3,

    # Output
    "output_dir": "./model_output",
    "model_name": "indobert-finansial-sentiment",
}

LABEL_NAMES = {0: "Negatif/Bearish", 1: "Netral", 2: "Positif/Bullish"}
os.makedirs(CONFIG["output_dir"], exist_ok=True)

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["random_seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {DEVICE}")


# ─────────────────────────────────────────────
# 1. DATASET CLASS
# ─────────────────────────────────────────────
class FinancialSentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = int(self.labels[idx])

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_attention_mask=True,
            return_tensors="pt",
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "token_type_ids": encoding["token_type_ids"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long),
        }


# ─────────────────────────────────────────────
# 2. LOAD & SPLIT DATA
# ─────────────────────────────────────────────
def load_data():
    print("\n📂 Loading dataset...")
    df = pd.read_csv(CONFIG["data_path"], sep=",", on_bad_lines="skip", engine="python")
    print(f"   Loaded columns: {df.columns.tolist()}") # Added for debugging
    df = df[[CONFIG["text_col"], CONFIG["label_col"]]].dropna()
    df[CONFIG["label_col"]] = df[CONFIG["label_col"]].astype(int)

    print(f"   Total samples : {len(df)}")
    print(f"   Label dist    :\n{df[CONFIG['label_col']].value_counts().to_string()}")

    X = df[CONFIG["text_col"]]
    y = df[CONFIG["label_col"]]

    # Stratified split → train / val / test
    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y,
        test_size=CONFIG["test_size"] + CONFIG["val_size"],
        stratify=y,
        random_state=CONFIG["random_seed"],
    )
    val_ratio = CONFIG["val_size"] / (CONFIG["test_size"] + CONFIG["val_size"])
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp,
        test_size=1 - val_ratio,
        stratify=y_temp,
        random_state=CONFIG["random_seed"],
    )

    print(f"\n   Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}")
    return X_train, X_val, X_test, y_train, y_val, y_test


# ─────────────────────────────────────────────
# 3. TRAINING & EVALUATION
# ─────────────────────────────────────────────
def train_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    for batch in loader:
        optimizer.zero_grad()
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        token_type_ids = batch["token_type_ids"].to(device)
        labels         = batch["labels"].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels,
        )
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), CONFIG["gradient_clip"])
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return total_loss / len(loader), correct / total


def eval_epoch(model, loader, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            token_type_ids = batch["token_type_ids"].to(device)
            labels         = batch["labels"].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                token_type_ids=token_type_ids,
                labels=labels,
            )
            total_loss += outputs.loss.item()
            preds = torch.argmax(outputs.logits, dim=1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    return total_loss / len(loader), correct / total, all_preds, all_labels


# ─────────────────────────────────────────────
# 4. PLOT HELPERS
# ─────────────────────────────────────────────
def plot_training_history(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    epochs = range(1, len(history["train_loss"]) + 1)

    axes[0].plot(epochs, history["train_loss"], "b-o", label="Train Loss")
    axes[0].plot(epochs, history["val_loss"],   "r-o", label="Val Loss")
    axes[0].set_title("Loss per Epoch"); axes[0].set_xlabel("Epoch")
    axes[0].set_ylabel("Loss"); axes[0].legend(); axes[0].grid(True)

    axes[1].plot(epochs, history["train_acc"], "b-o", label="Train Acc")
    axes[1].plot(epochs, history["val_acc"],   "r-o", label="Val Acc")
    axes[1].set_title("Accuracy per Epoch"); axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("Accuracy"); axes[1].legend(); axes[1].grid(True)

    plt.tight_layout()
    path = os.path.join(CONFIG["output_dir"], "training_history.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"   📊 Training history saved → {path}")


def plot_confusion_matrix(y_true, y_pred, split_name="Test"):
    cm = confusion_matrix(y_true, y_pred)
    labels = [LABEL_NAMES[i] for i in range(CONFIG["num_labels"])]
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=labels, yticklabels=labels)
    plt.title(f"Confusion Matrix — {split_name} Set")
    plt.ylabel("True Label"); plt.xlabel("Predicted Label")
    plt.tight_layout()
    path = os.path.join(CONFIG["output_dir"], f"confusion_matrix_{split_name.lower()}.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"   📊 Confusion matrix saved → {path}")


# ─────────────────────────────────────────────
# 5. MAIN
# ─────────────────────────────────────────────
def main():
    # ── Load data ──────────────────────────────
    X_train, X_val, X_test, y_train, y_val, y_test = load_data()

    # ── Tokenizer ──────────────────────────────
    print(f"\n🔤 Loading tokenizer: {CONFIG['pretrained_model']}")
    tokenizer = BertTokenizer.from_pretrained(CONFIG["pretrained_model"])

    train_ds = FinancialSentimentDataset(X_train, y_train, tokenizer, CONFIG["max_length"])
    val_ds   = FinancialSentimentDataset(X_val,   y_val,   tokenizer, CONFIG["max_length"])
    test_ds  = FinancialSentimentDataset(X_test,  y_test,  tokenizer, CONFIG["max_length"])

    train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)
    test_loader  = DataLoader(test_ds,  batch_size=CONFIG["batch_size"], shuffle=False, num_workers=0)

    # ── Model ──────────────────────────────────
    print(f"\n🤖 Loading model: {CONFIG['pretrained_model']}")
    model = BertForSequenceClassification.from_pretrained(
        CONFIG["pretrained_model"],
        num_labels=CONFIG["num_labels"],
        ignore_mismatched_sizes=True,
    )
    model.to(DEVICE)

    # ── Optimizer & Scheduler ─────────────────
    no_decay = ["bias", "LayerNorm.weight"]
    optimizer_grouped_parameters = [
        {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
         "weight_decay": CONFIG["weight_decay"]},
        {"params": [p for n, p in model.named_parameters() if     any(nd in n for nd in no_decay)],
         "weight_decay": 0.0},
    ]
    optimizer = AdamW(optimizer_grouped_parameters, lr=CONFIG["learning_rate"])

    total_steps  = len(train_loader) * CONFIG["epochs"]
    warmup_steps = int(total_steps * CONFIG["warmup_ratio"])
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
    )

    # ── Training loop ─────────────────────────
    print(f"\n🚀 Starting training — {CONFIG['epochs']} epochs on {DEVICE}\n{'─'*55}")
    history     = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
    best_val_f1 = 0.0
    patience_ctr = 0

    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
        val_loss,   val_acc, val_preds, val_labels = eval_epoch(model, val_loader, DEVICE)
        val_f1 = f1_score(val_labels, val_preds, average="macro")

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch {epoch}/{CONFIG['epochs']}  "
              f"| Train Loss: {train_loss:.4f}  Acc: {train_acc:.4f}  "
              f"| Val Loss: {val_loss:.4f}  Acc: {val_acc:.4f}  F1: {val_f1:.4f}")

        # Save best model
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            patience_ctr = 0
            save_path = os.path.join(CONFIG["output_dir"], CONFIG["model_name"])
            model.save_pretrained(save_path)
            tokenizer.save_pretrained(save_path)
            print(f"   ✅ Best model saved (val F1={best_val_f1:.4f}) → {save_path}")
        else:
            patience_ctr += 1
            if patience_ctr >= CONFIG["early_stopping_patience"]:
                print(f"\n⚠️  Early stopping triggered at epoch {epoch}")
                break

    # ── Plots ─────────────────────────────────
    print("\n📊 Generating plots...")
    plot_training_history(history)
    plot_confusion_matrix(val_labels, val_preds, split_name="Validation")

    # ── Test Evaluation ───────────────────────
    print("\n🧪 Evaluating on Test Set...")
    best_model = BertForSequenceClassification.from_pretrained(
        os.path.join(CONFIG["output_dir"], CONFIG["model_name"])
    ).to(DEVICE)
    _, test_acc, test_preds, test_labels = eval_epoch(best_model, test_loader, DEVICE)

    target_names = [LABEL_NAMES[i] for i in range(CONFIG["num_labels"])]
    print("\n" + "─"*55)
    print("📋 CLASSIFICATION REPORT — TEST SET")
    print("─"*55)
    print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))

    plot_confusion_matrix(test_labels, test_preds, split_name="Test")

    # Save summary
    report_dict = classification_report(
        test_labels, test_preds, target_names=target_names, output_dict=True
    )
    summary = {
        "model": CONFIG["pretrained_model"],
        "test_accuracy": accuracy_score(test_labels, test_preds),
        "macro_f1": f1_score(test_labels, test_preds, average="macro"),
        "weighted_f1": f1_score(test_labels, test_preds, average="weighted"),
        "best_val_f1": best_val_f1,
        "epochs_trained": epoch,
        "per_class": {k: v for k, v in report_dict.items() if k in target_names},
    }
    import json
    summary_path = os.path.join(CONFIG["output_dir"], "eval_summary.json")
    with open(summary_path, "w") as f:
        json.dump(summary, f, indent=2)
    print(f"\n📄 Eval summary saved → {summary_path}")
    print(f"\n🏁 Done! Model path: {os.path.join(CONFIG['output_dir'], CONFIG['model_name'])}")


if __name__ == "__main__":
    main()